# skrald

En notebook til datarensning og hurtig analyse.

In [58]:
from simulated_city.maplibre_live import LiveMapLibreMap
from IPython.display import display

# Input (lat, lng)
LAT = 55.64151318863417
LNG = 12.104264908902627

# anymap bruger (lng, lat)
m = LiveMapLibreMap(
    center=(LNG, LAT),
    zoom=13.5,
    height="500px",
    width="100%",
)
m.add_basemap("OpenStreetMap.Mapnik")
m.add_3d_buildings()

display(m)

In [61]:
import random
import time

# Sensorpunkter / husstande (lat, lng, navn, marker-id)
households = [
    (55.647662449699915, 12.102173996954603, "Husstand 1", "household-1"),
    (55.645890751659195, 12.090165719302282, "Husstand 2", "household-2"),
    (55.63765288266809, 12.081024983597791, "Husstand 3", "household-3"),
    (55.636737425843485, 12.102416347986289, "Husstand 4", "household-4"),
    (55.63559867257449, 12.108683544261638, "Husstand 5", "household-5"),
]

# Rødt punkt til højre = Argo (depot/garage for skraldebil)
ARGO_LAT = 55.640705843486664
ARGO_LNG = 12.121046368480554
ARGO_ID = "Argo"

GREEN = "#39A53F"
BROWN = "#6D4C41"
RED = "#D32F2F"
TRUCK_GRAY = "#9E9E9E"

# Print kun detaljeret status for Husstand 1
WATCH_MARKER_ID = "household-1"

# Faste "knækkede" ruter pr. husstand (lng, lat)
CUSTOM_ROUTES = {
    "household-1": [
        (12.120405652879136, 55.64288745340195),
        (12.104161407476502, 55.64153617866285),
        (12.104761514065949, 55.643750986384276),
        (12.102036700016578, 55.64765792042802),
    ],
    "household-2": [
        (12.120405652879136, 55.64288745340195),
        (12.092551571564814, 55.640521328299215),
        (12.092606003048006, 55.64159799285009),
    ],
    "household-3": [
        (12.120405652879136, 55.64288745340195),
        (12.094622764978585, 55.6406136437756),
        (12.09445925796891, 55.64021373890484),
        (12.093042160034736, 55.639783054275924),
        (12.092769721239756, 55.638491048390335),
        (12.089445124478328, 55.63812178609732),
        (12.082414925152499, 55.6362139694269),
    ],
    "household-4": [
        (12.120405652879136, 55.64288745340195),
        (12.104270435101448, 55.641597694433464),
    ],
    "household-5": [
        (12.120405652879136, 55.64288745340195),
        (12.116480647941305, 55.64248828512258),
        (12.114680508290075, 55.63978151446282),
        (12.110973567874579, 55.638336234993005),
        (12.109556076270385, 55.63716744948747),
    ],
}

# Simulations-parametre
DISPATCH_DELAY_SECONDS = 1.0   # Vent 1 sek efter brun, før bilen sendes
DISPATCH_SECONDS = 7.0         # Linje (ud + retur) vises i 7 sek
COOLDOWN_SECONDS = 2.0         # Efter tømning: 2 sek countdown
RANGE_MIN = 7.0                # Ny fyldningstid min
RANGE_MAX = 17.0               # Ny fyldningstid max
FORCE_WITHIN_SECONDS = 10.0    # Min. ét grønt punkt skal aktivere indenfor 10 sek
SIMULATION_SECONDS = 90.0      # Kør i 90 sek (juster efter behov)

def random_fill_seconds():
    return random.uniform(RANGE_MIN, RANGE_MAX)

def ensure_activation_within_10(state, now):
    green_items = [item for item in state.values() if item["phase"] == "green"]
    if not green_items:
        return

    earliest_next_brown = min(item["next_brown_at"] for item in green_items)
    if earliest_next_brown <= now + FORCE_WITHIN_SECONDS:
        return

    forced_item = random.choice(green_items)
    forced_delay = random.uniform(RANGE_MIN, FORCE_WITHIN_SECONDS)
    forced_item["next_brown_at"] = now + forced_delay

def log_house_1(marker_id, message):
    if marker_id == WATCH_MARKER_ID:
        print(message)

def interpolate_on_route(route_coords, progress):
    if not route_coords:
        return (ARGO_LNG, ARGO_LAT)

    progress = min(max(progress, 0.0), 1.0)
    if len(route_coords) == 1:
        return tuple(route_coords[0])

    segment_lengths = []
    total = 0.0
    for i in range(len(route_coords) - 1):
        x1, y1 = route_coords[i]
        x2, y2 = route_coords[i + 1]
        seg_len = ((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5
        segment_lengths.append(seg_len)
        total += seg_len

    if total == 0.0:
        return tuple(route_coords[-1])

    target = progress * total
    traversed = 0.0
    for i, seg_len in enumerate(segment_lengths):
        if traversed + seg_len >= target:
            local = (target - traversed) / seg_len if seg_len > 0 else 0.0
            x1, y1 = route_coords[i]
            x2, y2 = route_coords[i + 1]
            lng = x1 + (x2 - x1) * local
            lat = y1 + (y2 - y1) * local
            return (lng, lat)
        traversed += seg_len

    return tuple(route_coords[-1])

# Ryd op fra tidligere kørsel af celle 3 (gamle punkter/streger)
if "simulation_marker_ids" in globals():
    for old_marker_id in simulation_marker_ids:
        try:
            m.remove_marker(old_marker_id)
        except Exception:
            pass

if "simulation_line_ids" in globals():
    for old_line_id in simulation_line_ids:
        try:
            m.set_visibility(old_line_id, False)
        except Exception:
            pass

for legacy_marker_id in [
    "dispatch-center", "Argo",
    "household-1", "household-2", "household-3", "household-4", "household-5",
]:
    try:
        m.remove_marker(legacy_marker_id)
    except Exception:
        pass

simulation_marker_ids = [ARGO_ID, "household-1", "household-2", "household-3", "household-4", "household-5"]
simulation_line_ids = []

# Opret Argo-punkt (rød)
m.add_marker(
    ARGO_LNG,
    ARGO_LAT,
    name=ARGO_ID,
    color=RED,
    popup="🚛 Argo: sender skraldebil ved signal fra brun husstand",
)

# Tilstand pr. husstand
state = {}
now = time.time()
for lat, lng, house_name, marker_id in households:
    fill_seconds = random_fill_seconds()
    m.add_marker(
        lng,
        lat,
        name=marker_id,
        color=GREEN,
        popup=f"{house_name}<br>📡 Grøn: fyldes op<br>Bliver brun om ca. {fill_seconds:.1f}s",
    )
    state[marker_id] = {
        "name": house_name,
        "lat": lat,
        "lng": lng,
        "phase": "green",                  # green | brown_wait | dispatch | cooldown
        "next_brown_at": now + fill_seconds,
        "dispatch_start_at": None,
        "dispatch_started_at": None,
        "dispatch_end_at": None,
        "cooldown_end_at": None,
        "line_layer_id": None,
        "truck_marker_id": None,
        "dispatch_route": None,
        "last_countdown": None,
    }

ensure_activation_within_10(state, now)

display(m)

sim_start = time.time()
while time.time() - sim_start < SIMULATION_SECONDS:
    now = time.time()
    ensure_activation_within_10(state, now)

    for marker_id, item in state.items():
        phase = item["phase"]

        # 1) Grøn -> Brun (signal sendt, men vent 1 sek før dispatch-linje)
        if phase == "green" and now >= item["next_brown_at"]:
            m.remove_marker(marker_id)
            m.add_marker(
                item["lng"],
                item["lat"],
                name=marker_id,
                color=BROWN,
                popup=f"{item['name']}<br>🟤 Brun: fyldt op<br>Signal sendt til Argo<br>Dispatch om 1 sek",
            )
            item["phase"] = "brown_wait"
            item["dispatch_start_at"] = now + DISPATCH_DELAY_SECONDS
            log_house_1(marker_id, f"Signal: {item['name']} er brun -> Argo klargør skraldebil")
            continue

        # 2) Efter 1 sek: send dispatch-linje i 7 sek + visuel skraldebil
        if phase == "brown_wait" and now >= item["dispatch_start_at"]:
            line_layer_id = f"dispatch-line-{marker_id}-{int(now * 1000)}"

            outward = CUSTOM_ROUTES.get(marker_id, [])
            outward = outward + [(item["lng"], item["lat"])]  # sikrer at ruten rammer huset

            outward_coords = [[lng, lat] for lng, lat in outward]
            return_coords = [[lng, lat] for lng, lat in reversed(outward[:-1])]

            coordinates = [[ARGO_LNG, ARGO_LAT]] + outward_coords + return_coords + [[ARGO_LNG, ARGO_LAT]]

            route_geojson = {
                "type": "FeatureCollection",
                "features": [
                    {
                        "type": "Feature",
                        "geometry": {
                            "type": "LineString",
                            "coordinates": coordinates,
                        },
                        "properties": {},
                    }
                ],
            }

            m.add_geojson(
                route_geojson,
                name=line_layer_id,
                layer_type="line",
                paint={
                    "line-color": RED,
                    "line-width": 4,
                    "line-opacity": 0.9,
                },
                fit_bounds=False,
            )
            simulation_line_ids.append(line_layer_id)

            truck_marker_id = f"truck-{marker_id}"
            m.add_marker(
                ARGO_LNG,
                ARGO_LAT,
                name=truck_marker_id,
                color=TRUCK_GRAY,
                popup=f"🚚 Skraldebil: Argo → {item['name']} → Argo",
            )
            simulation_marker_ids.append(truck_marker_id)

            item["phase"] = "dispatch"
            item["dispatch_started_at"] = now
            item["dispatch_end_at"] = now + DISPATCH_SECONDS
            item["line_layer_id"] = line_layer_id
            item["truck_marker_id"] = truck_marker_id
            item["dispatch_route"] = coordinates
            log_house_1(marker_id, f"Argo sender skraldebil til {item['name']}")
            continue

        # 3) Under dispatch: flyt visuel skraldebil langs ruten
        if phase == "dispatch":
            if item["dispatch_started_at"] is not None and item["dispatch_route"]:
                progress = (now - item["dispatch_started_at"]) / DISPATCH_SECONDS
                truck_lng, truck_lat = interpolate_on_route(item["dispatch_route"], progress)
                m.move_marker(
                    item["truck_marker_id"],
                    (truck_lng, truck_lat),
                    color=TRUCK_GRAY,
                    popup=f"🚚 Skraldebil i drift: {item['name']}",
                )

            # Dispatch færdig -> skjul linje + fjern bil + start 2 sek cooldown
            if now >= item["dispatch_end_at"]:
                if item["line_layer_id"]:
                    m.set_visibility(item["line_layer_id"], False)
                if item["truck_marker_id"]:
                    try:
                        m.remove_marker(item["truck_marker_id"])
                    except Exception:
                        pass

                item["phase"] = "cooldown"
                item["cooldown_end_at"] = now + COOLDOWN_SECONDS
                item["last_countdown"] = None
                log_house_1(marker_id, f"Hus tømt: {item['name']}")
                log_house_1(marker_id, f"Skraldebil vender retur fra {item['name']} til Argo")
            continue

        # 4) Cooldown countdown -> tilbage til grøn med ny tilfældig 7-17 sek
        if phase == "cooldown":
            remaining = max(0.0, item["cooldown_end_at"] - now)
            remaining_sec = int(remaining + 0.999)

            if item["last_countdown"] != remaining_sec and remaining_sec > 0:
                m.remove_marker(marker_id)
                m.add_marker(
                    item["lng"],
                    item["lat"],
                    name=marker_id,
                    color=BROWN,
                    popup=f"{item['name']}<br>♻️ Tømt<br>Ny cyklus starter om {remaining_sec}s",
                )
                item["last_countdown"] = remaining_sec

            if remaining <= 0:
                fill_seconds = random_fill_seconds()
                m.remove_marker(marker_id)
                m.add_marker(
                    item["lng"],
                    item["lat"],
                    name=marker_id,
                    color=GREEN,
                    popup=f"{item['name']}<br>📡 Grøn: fyldes op igen<br>Bliver brun om ca. {fill_seconds:.1f}s",
                )
                item["phase"] = "green"
                item["next_brown_at"] = now + fill_seconds
                item["dispatch_start_at"] = None
                item["dispatch_started_at"] = None
                item["dispatch_end_at"] = None
                item["cooldown_end_at"] = None
                item["line_layer_id"] = None
                item["truck_marker_id"] = None
                item["dispatch_route"] = None
                item["last_countdown"] = None
                log_house_1(marker_id, f"{item['name']} er nulstillet -> ny fyldning starter")

    time.sleep(0.2)

print("Simulation færdig (90 sek). Kør celle 3 igen for en ny tilfældig cyklus.")

KeyboardInterrupt: 